# 11 · NumPy versus PyTorch: what frameworks do for you

## What you will build

You have built every part of a neural network by hand. Real projects use libraries instead: scikit-learn for quick standard models, and frameworks such as PyTorch for anything larger. Here you rebuild your model with them and compare, on exactly the same data split:

- the benchmark: your own NumPy network, timed;
- the same architecture as a scikit-learn `MLPClassifier`;
- the same model in PyTorch, with a check that its automatic differentiation computes exactly the gradients your `backward` computes (this part runs where PyTorch is installed, for example in Google Colab);
- your own mini-framework in the style frameworks use, with three classes you type yourself: `Linear`, `ReLU` and `Sequential`, each with `forward`, `backward` and `params`.

## Learning objectives

- Map each piece of your NumPy network to its counterpart in scikit-learn and in PyTorch.
- Compare accuracy, training time and code size fairly: one split, one architecture, one training recipe.
- Explain what automatic differentiation (autograd) automates, why optimised kernels and GPUs matter for large models but not for tiny ones, and what a device is.
- Write layers as objects with `forward`, `backward` and `params`, and show that they reproduce your functional network exactly.

## Dependencies

Tier 1: NumPy and Matplotlib for every core cell. Two kinds of optional cells are tagged `comparison`, and nothing later depends on them:

- **scikit-learn**, which JupyterLite, the course JupyterHub server and Google Colab all provide;
- **PyTorch**, which Google Colab provides but JupyterLite and the course server do not. There the cell prints a note and skips itself; use "Open in Colab" to run it.

## Setup

Run this cell first. It loads NumPy and Matplotlib and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import matplotlib.pyplot as plt

## 📦 Recap: what you built earlier

The hidden cell below defines the pieces from earlier notebooks under their usual names, so this notebook runs on its own. Open it if you want to see them again.

It also contains `train_val_test_split`, `fit_standardizer` and `apply_standardizer` from notebook 09.

In [ ]:
# 📦 Recap: the pieces from notebooks 01–04, under their usual names.

def relu(Z):
    """ReLU, element by element: max(0, Z)."""
    return np.maximum(0.0, Z)


def relu_grad(Z):
    """The slope of ReLU: 1.0 where Z > 0, else 0.0 (we take 0 at Z = 0)."""
    return (Z > 0).astype(float)


def sigmoid(Z):
    """Numerically stable logistic function 1 / (1 + exp(-Z)), element by element."""
    return np.exp(-np.logaddexp(0.0, -Z))


def softmax(Z):
    """Row-wise softmax of logits Z (batch, classes). Subtracting each row's maximum avoids overflow."""
    exps = np.exp(Z - Z.max(axis=1, keepdims=True))
    return exps / exps.sum(axis=1, keepdims=True)


def dense_forward(X, W, b):
    """Z = X @ W + b. X: (batch, n_in), W: (n_in, n_out), b: (n_out,) -> Z: (batch, n_out)."""
    return X @ W + b


def init_network(sizes, rng):
    """He-initialised layers for sizes such as [2, 64, 3]: a list of {"W": (n_in, n_out), "b": (n_out,)}."""
    return [
        {"W": rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out)), "b": np.zeros(n_out)}
        for n_in, n_out in zip(sizes[:-1], sizes[1:])
    ]


def forward(network, X):
    """ReLU hidden layers and a softmax output.

    Returns (probs, cache): probs has shape (batch, classes), and cache[i] = (A_prev, Z) for layer i.
    """
    cache = []
    A = X
    for index, layer in enumerate(network):
        Z = dense_forward(A, layer["W"], layer["b"])
        cache.append((A, Z))
        A = softmax(Z) if index == len(network) - 1 else relu(Z)
    return A, cache


def predict(network, X):
    """The predicted class index for every row of X: shape (batch,)."""
    probs, _ = forward(network, X)
    return probs.argmax(axis=1)


def accuracy(network, X, y):
    """The fraction of rows of X whose predicted class equals the label in y."""
    return float(np.mean(predict(network, X) == y))


def make_spirals(n_per_class, n_classes, noise, rng):
    """n_classes interleaved spiral arms in 2-D, sorted by class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,). noise is the spread of the random angle jitter.
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    X_parts, y_parts = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 6.0 * radius + rng.normal(0.0, noise, n_per_class)
        X_parts.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        y_parts.append(np.full(n_per_class, k))
    return np.concatenate(X_parts), np.concatenate(y_parts)


# From notebook 05: loss functions.

def mse(pred, target):
    """Mean squared error over every entry of two arrays with the same shape."""
    return float(np.mean((pred - target) ** 2))


def binary_cross_entropy(p, y):
    """Mean binary cross-entropy of probabilities p (batch,) against 0/1 labels y (batch,), clipped."""
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def cross_entropy(probs, y):
    """Mean negative log-probability of the true class. probs: (batch, classes), y: (batch,) ints."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(np.clip(picked, 1e-12, None))))


def cross_entropy_from_logits(Z, y):
    """The same loss computed from logits Z (batch, classes) with the log-sum-exp trick."""
    row_max = Z.max(axis=1, keepdims=True)
    log_sum_exp = row_max[:, 0] + np.log(np.exp(Z - row_max).sum(axis=1))
    return float(np.mean(log_sum_exp - Z[np.arange(len(y)), y]))


# From notebook 06: numerical derivatives and gradient checking.

def numerical_derivative(f, x, h=1e-5):
    """Central-difference estimate of f'(x) for a function of one number."""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, array, eps=1e-5):
    """Central-difference gradient of the scalar f(array) with respect to every entry of array.

    Each entry is nudged in place and then restored, so f may also read the array
    through a closure (for example a network's weights). Returns an array shaped like array.
    """
    grad = np.zeros(array.shape)
    for index in np.ndindex(array.shape):
        original = array[index]
        array[index] = original + eps
        loss_plus = f(array)
        array[index] = original - eps
        loss_minus = f(array)
        array[index] = original
        grad[index] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def relative_error(a, b):
    """||a - b|| / (||a|| + ||b||), with a tiny floor on the denominator: 0 when equal, at most 1."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

# From notebook 07: backpropagation.

def softmax_cross_entropy_grad(probs, y):
    """dL/dZ for softmax + mean cross-entropy: (P - Y) / n."""
    return (probs - np.eye(probs.shape[1])[y]) / len(y)


def dense_backward(dZ, A_prev, W):
    """Gradients of a dense layer: (dW, db, dA_prev)."""
    return A_prev.T @ dZ, dZ.sum(axis=0), dZ @ W.T


def backward(network, cache, probs, y):
    """Every layer's gradients, last layer first, returned in layer order."""
    grads = [None] * len(network)
    dZ = softmax_cross_entropy_grad(probs, y)
    for i in reversed(range(len(network))):
        A_prev, Z = cache[i]
        if i < len(network) - 1:
            dZ = dA * relu_grad(Z)
        dW, db, dA = dense_backward(dZ, A_prev, network[i]["W"])
        grads[i] = {"W": dW, "b": db}
    return grads


# From notebook 08: training with gradient descent.

def sgd_step(network, grads, lr):
    """A new network one gradient-descent step downhill: W - lr * dW, b - lr * db."""
    return [{"W": layer["W"] - lr * g["W"], "b": layer["b"] - lr * g["b"]} for layer, g in zip(network, grads)]


def minibatch_indices(n, batch_size, seed):
    """A shuffled split of range(n) into batches of batch_size (the last may be smaller)."""
    order = np.random.default_rng(seed).permutation(n)
    return [order[start:start + batch_size] for start in range(0, n, batch_size)]


def train_epoch(network, X, y, lr, batch_size, seed):
    """One pass over the data in shuffled mini-batches. Returns (network, mean batch loss)."""
    losses = []
    for batch in minibatch_indices(len(X), batch_size, seed):
        probs, cache = forward(network, X[batch])
        losses.append(cross_entropy(probs, y[batch]))
        network = sgd_step(network, backward(network, cache, probs, y[batch]), lr)
    return network, float(np.mean(losses))


def train(network, X, y, X_val, y_val, lr, epochs, batch_size, seed=0):
    """Train for some epochs; returns (network, history of losses and accuracies per epoch)."""
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        network, _ = train_epoch(network, X, y, lr, batch_size, seed + epoch)
        history["train_loss"].append(cross_entropy(forward(network, X)[0], y))
        history["val_loss"].append(cross_entropy(forward(network, X_val)[0], y_val))
        history["train_acc"].append(accuracy(network, X, y))
        history["val_acc"].append(accuracy(network, X_val, y_val))
    return network, history


# From notebook 09: splitting and scaling.

def train_val_test_split(X, y, fractions, seed):
    """Stratified split into training, validation and test sets, class by class."""
    rng = np.random.default_rng(seed)
    parts = ([], [], [])
    for label in np.unique(y):
        rows = rng.permutation(np.flatnonzero(y == label))
        n_train = int(round(fractions[0] * len(rows)))
        n_val = int(round(fractions[1] * len(rows)))
        for part, chunk in zip(parts, (rows[:n_train], rows[n_train:n_train + n_val], rows[n_train + n_val:])):
            part.append(chunk)
    train, val, test = (np.concatenate(part) for part in parts)
    return X[train], y[train], X[val], y[val], X[test], y[test]


def fit_standardizer(X_train):
    """Column means and standard deviations from the training set only; zero stds become 1.0."""
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    return mean, np.where(std == 0, 1.0, std)


def apply_standardizer(X, mean, std):
    """Scale X (n, d) with training statistics mean (d,) and std (d,)."""
    return (X - mean) / std

## 1 · The benchmark: your NumPy network

A fair comparison changes one thing at a time. Every model below gets the same data split, the same architecture, `[2, 64, 64, 3]`, and, wherever the library allows it, the same training recipe: plain mini-batch gradient descent with a learning rate of 0.1, batches of 32 and 100 epochs. The data are 900 spiral points with moderate noise, split 60/20/20 and standardised with training statistics.

In [ ]:
X_all, y_all = make_spirals(300, 3, 0.35, np.random.default_rng(0))
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(X_all, y_all, (0.6, 0.2, 0.2), 1)
mean, std = fit_standardizer(X_train)
X_train, X_val, X_test = (apply_standardizer(part, mean, std) for part in (X_train, X_val, X_test))
SIZES, LR, EPOCHS, BATCH_SIZE, SEED = [2, 64, 64, 3], 0.1, 100, 32, 0
print("training", X_train.shape, "validation", X_val.shape, "test", X_test.shape)

In [ ]:
import time

start_network = init_network(SIZES, np.random.default_rng(SEED))
started = time.perf_counter()
numpy_network, numpy_history = train(start_network, X_train, y_train, X_val, y_val, LR, EPOCHS, BATCH_SIZE, SEED)
numpy_seconds = time.perf_counter() - started
numpy_test_accuracy = accuracy(numpy_network, X_test, y_test)
NUMPY_LINES = 65  # lines of code in the recap functions that define and train this network
updates = len(minibatch_indices(len(X_train), BATCH_SIZE, 0)) * EPOCHS
print(f"your NumPy network: test accuracy {numpy_test_accuracy:.3f}, {updates} updates in {numpy_seconds:.2f} s")

**Expected behaviour.** About 0.98 test accuracy after 1,700 updates, in a few tenths of a second. The model is defined and trained by 65 lines of your own code, counted from the recap: `relu` and `relu_grad`, `softmax`, `dense_forward`, `init_network`, `forward`, `predict`, `accuracy`, `cross_entropy`, `softmax_cross_entropy_grad`, `dense_backward`, `backward`, `sgd_step`, `minibatch_indices`, `train_epoch` and `train`, without blank lines, comments or docstrings.

## 2 · The same model in scikit-learn

scikit-learn's `MLPClassifier` is a multilayer perceptron, the same kind of network as yours. Its arguments map onto your code almost one to one:

| Your NumPy code | `MLPClassifier` |
|---|---|
| `SIZES = [2, 64, 64, 3]` | `hidden_layer_sizes=(64, 64)`; the input and output sizes come from the data |
| `relu` in the hidden layers | `activation="relu"` |
| `sgd_step` with a constant rate | `solver="sgd"`, `learning_rate="constant"`, `learning_rate_init=0.1`, `momentum=0.0` |
| `minibatch_indices` | `batch_size=32`, `shuffle=True` |
| `EPOCHS = 100` | `max_iter=100`, with `n_iter_no_change=100` so that it never stops early |
| no regularisation | `alpha=0.0` (the default, 0.0001, adds an L2 penalty) |
| He initialisation | its own uniform initialisation in the style of Glorot and Bengio, which you cannot switch off |
| softmax and cross-entropy | the same for three or more classes (two classes get a single sigmoid output) |

So the two models differ only in their starting weights and their shuffling. The cell below is tagged `comparison`: it needs scikit-learn, and nothing later depends on it.

In [ ]:
# Optional comparison: needs scikit-learn (JupyterLite, the course server and Colab all have it).
import warnings

from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier

sklearn_model = MLPClassifier(hidden_layer_sizes=(64, 64), activation="relu", solver="sgd", learning_rate="constant",
                              learning_rate_init=LR, momentum=0.0, nesterovs_momentum=False, alpha=0.0,
                              batch_size=BATCH_SIZE, max_iter=EPOCHS, n_iter_no_change=EPOCHS, tol=0.0,
                              shuffle=True, random_state=SEED)
started = time.perf_counter()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)  # it warns because we stop at max_iter on purpose
    sklearn_model.fit(X_train, y_train)
sklearn_seconds = time.perf_counter() - started
sklearn_test_accuracy = sklearn_model.score(X_test, y_test)
SKLEARN_LINES = 4  # the import, the constructor, fit and score

print(f"{'':>14}{'test accuracy':>15}{'training time':>15}{'lines of code':>15}")
print(f"{'your NumPy':>14}{numpy_test_accuracy:>15.3f}{numpy_seconds:>13.2f} s{NUMPY_LINES:>15d}")
print(f"{'scikit-learn':>14}{sklearn_test_accuracy:>15.3f}{sklearn_seconds:>13.2f} s{SKLEARN_LINES:>15d}")

**What to notice.** Both models reach about the same test accuracy (about 0.98 and 0.97 here; the small difference comes from the different starting weights and shuffling, and would change with the seed). scikit-learn is **not faster**: its training loop runs the same NumPy matrix products as yours, plus input checks and bookkeeping, so it takes a similar fraction of a second. What it saves is code: 4 lines instead of 65, each of them tested by a large community. What it costs is control: you cannot change the initialisation, add a new kind of layer or inspect a gradient.

## 3 · The same model in PyTorch

PyTorch keeps the ideas you built and changes who does the work:

- **Tensors** are PyTorch's arrays, like NumPy arrays that can also live on a GPU and remember how they were computed.
- **Autograd** (automatic differentiation) records every operation of the forward pass in a graph. `loss.backward()` then walks that graph backwards and applies the chain rule, which is exactly what your `backward` does by hand, but for any model you can write, with no derivation needed.
- **Optimised kernels.** Each operation runs as compiled, multithreaded code. NumPy's matrix product is also compiled code, so for a single matrix product the two are similar; frameworks pull ahead by fusing operations and, above all, by running on accelerators.
- **Devices.** `model.to("cuda")` moves the parameters to an NVIDIA GPU (`"mps"` on recent Macs), where large matrix products run many times faster. Every tensor in one computation must be on the same device. A model as small as ours is faster on the CPU, because starting each GPU operation costs more than the arithmetic itself.
- **Precision.** PyTorch defaults to 32-bit floats and NumPy to 64-bit. We ask for `float64` below so that the numbers can be compared digit for digit.

| Your NumPy code | PyTorch |
|---|---|
| `W` with shape `(n_in, n_out)` | `torch.nn.Linear(n_in, n_out)` stores `weight` with shape `(n_out, n_in)`, which is `W.T` |
| `forward(network, X)` | `model(X)`, which returns logits; the softmax is folded into the loss |
| `cross_entropy(softmax(Z), y)` | `torch.nn.functional.cross_entropy(Z, y)`, which takes logits |
| `backward(network, cache, probs, y)` | `loss.backward()` |
| `sgd_step(network, grads, lr)` | `optimizer.step()` for `torch.optim.SGD`, which updates the parameters in place |
| nothing: your gradients are new arrays at every step | `optimizer.zero_grad()`, because PyTorch adds new gradients to the old ones until you clear them |

The cell below copies your starting weights into a PyTorch model, compares one batch's gradients with your `backward`, then trains on exactly the same mini-batches as your `train`. It is tagged `comparison` and skips itself where PyTorch is not installed.

In [ ]:
# Optional comparison: needs PyTorch. Google Colab has it; JupyterLite and the course server do not.
try:
    import torch
except ImportError:
    torch = None
    print("PyTorch is not installed here, so this comparison is skipped.")
    print('Use "Open in Colab" (PyTorch is preinstalled there) to run this cell.')

if torch is not None:
    def torch_model_from(network):
        """A float64 PyTorch copy of a NumPy network (torch.nn.Linear stores W transposed)."""
        layers = []
        for index, layer in enumerate(network):
            linear = torch.nn.Linear(*layer["W"].shape, dtype=torch.float64)
            with torch.no_grad():
                linear.weight.copy_(torch.from_numpy(layer["W"].T))
                linear.bias.copy_(torch.from_numpy(layer["b"]))
            layers.append(linear)
            if index < len(network) - 1:
                layers.append(torch.nn.ReLU())
        return torch.nn.Sequential(*layers)

    # 1. Autograd against your backward, on one mini-batch from the same starting weights.
    model = torch_model_from(start_network)
    batch = minibatch_indices(len(X_train), BATCH_SIZE, SEED)[0]
    loss = torch.nn.functional.cross_entropy(model(torch.from_numpy(X_train[batch])), torch.from_numpy(y_train[batch]))
    loss.backward()
    probs, cache = forward(start_network, X_train[batch])
    your_grads = backward(start_network, cache, probs, y_train[batch])
    linears = [module for module in model if isinstance(module, torch.nn.Linear)]
    gradient_gap = max(max(np.abs(linear.weight.grad.numpy().T - grad["W"]).max(),
                           np.abs(linear.bias.grad.numpy() - grad["b"]).max())
                       for linear, grad in zip(linears, your_grads))
    print(f"loss on one batch: PyTorch {loss.item():.12f}, yours {cross_entropy(probs, y_train[batch]):.12f}")
    print(f"largest difference between autograd and your backward: {gradient_gap:.1e}")

    # 2. Train on exactly the same mini-batches as your train function.
    model = torch_model_from(start_network)
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)
    X_t, y_t = torch.from_numpy(X_train), torch.from_numpy(y_train)
    started = time.perf_counter()
    for epoch in range(EPOCHS):
        for batch in minibatch_indices(len(X_train), BATCH_SIZE, SEED + epoch):
            rows = torch.from_numpy(batch)
            optimizer.zero_grad()
            torch.nn.functional.cross_entropy(model(X_t[rows]), y_t[rows]).backward()
            optimizer.step()
    torch_seconds = time.perf_counter() - started
    with torch.no_grad():
        torch_test_accuracy = float((model(torch.from_numpy(X_test)).argmax(dim=1).numpy() == y_test).mean())
    linears = [module for module in model if isinstance(module, torch.nn.Linear)]
    weight_gap = max(float(np.abs(linear.weight.detach().numpy().T - layer["W"]).max())
                     for linear, layer in zip(linears, numpy_network))
    print(f"after training: test accuracy PyTorch {torch_test_accuracy:.3f}, yours {numpy_test_accuracy:.3f}")
    print(f"largest weight difference after {EPOCHS} epochs: {weight_gap:.1e}")
    print(f"training time: PyTorch {torch_seconds:.2f} s, your NumPy {numpy_seconds:.2f} s")
    print("a CUDA GPU is", "available" if torch.cuda.is_available() else "not available", "here; this model ran on the CPU")

**What to notice.** Where PyTorch is missing, the cell prints a short note and nothing else breaks. Where it runs (in Colab, for example), the two losses agree to about 15 digits, and the largest gradient difference is around $10^{-16}$, the size of float64 rounding: autograd computed exactly the derivatives you derived in notebook 07. After 100 epochs on the same mini-batches, the weights still agree to within about $10^{-14}$ (the last few digits differ only because the two libraries add numbers in different orders), and the test accuracies are identical. PyTorch typically takes a few times longer than NumPy on this tiny model, because each of its operations carries more bookkeeping (the autograd graph); that overhead only pays off on larger models and on GPUs.

## 4 · 🧪 Where does the time go?

One training step costs two things: a fixed **overhead** for every operation (Python calls, creating arrays, checks) and the **arithmetic** itself, which grows with the layer sizes. A step of a `[2, w, w, 3]` network on a batch of 64 does about $3 \times 64 \times (2w + w^2 + 3w)$ multiply-adds: the forward products, plus twice as many in the backward pass. For small $w$ the overhead dominates, so the time hardly changes with the width; for large $w$ the $w^2$ term takes over. Optimised kernels and GPUs speed up the arithmetic, not the overhead, which is why they matter for big models and not for ours.

Time a few training steps for several widths (20 steps up to width 256 and 5 for wider layers, so that the cell stays quick). Add widths to the list and run the cell again.

In [ ]:
widths = [16, 64, 256, 1024]  # 🧪 try adding 4 or 2048 (2048 takes a few seconds)
batch_X, batch_y = X_train[:64], y_train[:64]
timings = []
for width in widths:
    network = init_network([2, width, width, 3], np.random.default_rng(0))
    steps = 20 if width <= 256 else 5
    started = time.perf_counter()
    for step in range(steps):
        probs, cache = forward(network, batch_X)
        network = sgd_step(network, backward(network, cache, probs, batch_y), 0.01)
    microseconds = (time.perf_counter() - started) / steps * 1e6
    multiply_adds = 3 * 64 * (2 * width + width * width + 3 * width)
    timings.append((width, microseconds))
    print(f"width {width:5d}: {microseconds:8.0f} µs per step, {multiply_adds / 1e6:7.2f} million multiply-adds, "
          f"{multiply_adds / microseconds:7.0f} multiply-adds per µs")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.loglog([w for w, _ in timings], [t for _, t in timings], "o-")
ax.set_xlabel("hidden width w")
ax.set_ylabel("µs per training step")
plt.show()

**What to notice.** Going from width 16 to 64 multiplies the arithmetic by more than 10 but the time by much less: the step is mostly overhead. From 256 to 1024 the arithmetic grows about 16 times and the time about 10 times: now the matrix products dominate, and the computer gets through far more multiply-adds per microsecond (about 80 times more than at width 16 on the machine used to test this notebook). This is the regime where frameworks, fused kernels and GPUs earn their keep.

## 5 · Build your own mini-framework

Frameworks organise a network as **layer objects**. Each layer keeps its own parameters and offers three methods:

- `forward(X)` computes the layer's output and **remembers** whatever the backward pass will need;
- `backward(d_out)` receives $\partial L/\partial(\text{output})$, stores the gradients of its own parameters and returns $\partial L/\partial(\text{input})$ for the layer before it;
- `params()` lists `(parameter, gradient)` pairs, so that an optimiser can update every parameter without knowing which layer it belongs to.

A `Sequential` container chains layers: forward in order, backward in reverse. That is the design of PyTorch's `torch.nn.Module` and `torch.nn.Sequential`, minus autograd: here each layer still knows its own derivative rule, exactly the rules you wrote in notebook 07.

`practice` checks functions. To check a class, `practice_class` runs a small script on an object of your class and on one of the reference class, compares everything the script returns and hands back whichever class passed.

In [ ]:
def practice_class(name, yours, reference, run, cases):
    """practice for a class: run(cls, *case) builds an object, uses it and returns what to compare.

    Returns your class once it matches the reference on every case, otherwise the reference class.
    """
    def run_yours(*case):
        return run(yours, *case)

    def run_reference(*case):
        return run(reference, *case)

    return yours if practice(name, run_yours, run_reference, cases) is run_yours else reference


def run_activation(layer_class, Z, dA):
    """Run forward and backward on a fresh activation layer; return the output, the input gradient and params."""
    layer = layer_class()
    return layer.forward(Z), layer.backward(dA), layer.params()

In [ ]:
layer_demo = {"W": np.array([[1.0, 2.0], [0.0, -1.0]]), "b": np.array([0.5, 0.0])}
X_demo = np.array([[1.0, 1.0]])
print("forward:", dense_forward(X_demo, layer_demo["W"], layer_demo["b"]))
print("backward for dZ = [[1, 1]]:", dense_backward(np.array([[1.0, 1.0]]), X_demo, layer_demo["W"]))

### ✍️ Your turn: `Linear`

Complete the three methods of `Linear`. `__init__` is written for you: it draws `self.W` `(n_in, n_out)` with He initialisation from `rng`, sets `self.b` `(n_out,)` to zeros and creates zero gradients `self.dW` and `self.db` with the same shapes.

- `forward(self, X)`: store the input as `self.X` (the backward pass needs it) and return `X @ self.W + self.b`, with shape `(batch, n_out)`.
- `backward(self, dZ)`: given `dZ` `(batch, n_out)`, store `self.dW = self.X.T @ dZ` and `self.db = dZ.sum(axis=0)`, and return `dZ @ self.W.T`, with shape `(batch, n_in)`.
- `params(self)`: return `[(self.W, self.dW), (self.b, self.db)]`.

Hint: these are your `dense_forward` and `dense_backward`, with the arrays now living on the object.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
class LinearReference:
    def __init__(self, n_in, n_out, rng):
        self.W = rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out))
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dZ):
        self.dW = self.X.T @ dZ
        self.db = dZ.sum(axis=0)
        return dZ @ self.W.T

    def params(self):
        return [(self.W, self.dW), (self.b, self.db)]

In [ ]:
class Linear:
    """A dense layer, Z = X @ W + b, that remembers its input for the backward pass."""

    def __init__(self, n_in, n_out, rng):
        self.W = rng.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out))  # He initialisation
        self.b = np.zeros(n_out)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, X):
        """X: (batch, n_in) -> Z: (batch, n_out). Store X as self.X for backward."""
        # ✍️ Your code here (two lines)
        raise NotImplementedError

    def backward(self, dZ):
        """dZ: (batch, n_out). Store self.dW and self.db; return dX: (batch, n_in)."""
        # ✍️ Your code here (three lines)
        raise NotImplementedError

    def params(self):
        """The (parameter, gradient) pairs, for the optimiser."""
        # ✍️ Your code here (one line)
        raise NotImplementedError

In [ ]:
def run_linear(layer_class, X, dZ):
    """Build a 3 -> 2 layer from a fixed seed, run forward and backward, and return what it produced."""
    layer = layer_class(3, 2, np.random.default_rng(0))
    Z = layer.forward(X)
    dX = layer.backward(dZ)
    return Z, dX, layer.params()


X_check = np.random.default_rng(1).normal(size=(4, 3))
dZ_check = np.random.default_rng(2).normal(size=(4, 2))
Linear = practice_class("Linear", Linear, LinearReference, run_linear, cases=[
    (X_check, dZ_check),
    (X_check[:1], dZ_check[:1]),    # edge case: a batch of one
])

**Expected behaviour.** The demo prints the forward output `[[1.5, 1.0]]` and the three backward results of your functional layer: `dW = [[1, 1], [1, 1]]`, `db = [1, 1]` and `dA_prev = [[3, -1]]`. Your class must produce the same numbers through its methods, and `params()` must pair each parameter with the gradient that `backward` just stored.

In [ ]:
Z_demo = np.array([[-1.0, 0.0, 2.0]])
print("mask:", Z_demo > 0, " output:", np.where(Z_demo > 0, Z_demo, 0.0), " relu_grad:", relu_grad(Z_demo))

### ✍️ Your turn: `ReLU`

Complete `forward` and `backward` of `ReLU`; `params` is written for you, because a ReLU has nothing to learn.

- `forward(self, Z)`: store the boolean mask `self.mask = Z > 0` and return `np.where(self.mask, Z, 0.0)`, which is `Z` where the mask is `True` and 0 elsewhere.
- `backward(self, dA)`: return `dA * self.mask`: the gradient passes only through the units that were active, exactly as `relu_grad` decided in notebook 07.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
class ReLUReference:
    def forward(self, Z):
        self.mask = Z > 0
        return np.where(self.mask, Z, 0.0)

    def backward(self, dA):
        return dA * self.mask

    def params(self):
        return []

In [ ]:
class ReLU:
    """max(0, Z), remembering where Z > 0 for the backward pass."""

    def forward(self, Z):
        """Store self.mask = Z > 0; return Z where the mask is True and 0 elsewhere."""
        # ✍️ Your code here (two lines)
        raise NotImplementedError

    def backward(self, dA):
        """Pass the gradient through where the unit was active."""
        # ✍️ Your code here (one line)
        raise NotImplementedError

    def params(self):
        return []

In [ ]:
Z_check = np.array([[-1.0, 0.0, 2.0], [3.0, -0.5, 0.25]])
ReLU = practice_class("ReLU", ReLU, ReLUReference, run_activation, cases=[
    (Z_check, np.ones_like(Z_check)),
    (Z_check, np.arange(6.0).reshape(2, 3)),    # the zero entry must block its gradient too
])

**Expected behaviour.** For the second case the output is `[[0, 0, 2], [3, 0, 0.25]]` and the gradient `[[0, 0, 2], [3, 0, 5]]`: the gradient survives exactly where the input was positive. At $Z = 0$ it is blocked, matching `relu_grad`.

In [ ]:
steps = [lambda v: v + 1, lambda v: v * 10]
value = 2
for step in steps:
    value = step(value)
print("apply in order, (2 + 1) * 10 =", value)

### ✍️ Your turn: `Sequential`

Complete the three methods of `Sequential`; `__init__` stores the layers in `self.layers`.

- `forward(self, X)`: pass `X` through every layer in order (`X = layer.forward(X)`) and return the result.
- `backward(self, d_out)`: pass the gradient through the layers in **reverse** order (`d_out = layer.backward(d_out)`) and return what comes out at the start: the gradient with respect to the input `X`.
- `params(self)`: one list holding the `(parameter, gradient)` pairs of every layer, in layer order.

Hints: `reversed(self.layers)`; for `params`, extend a list with each layer's `params()`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below if you get stuck.

In [ ]:
class SequentialReference:
    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, d_out):
        for layer in reversed(self.layers):
            d_out = layer.backward(d_out)
        return d_out

    def params(self):
        return [pair for layer in self.layers for pair in layer.params()]

In [ ]:
class Sequential:
    """Layers applied in order; gradients flow back through them in reverse order."""

    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, X):
        """Feed X through every layer in order and return the last output."""
        # ✍️ Your code here (three lines)
        raise NotImplementedError

    def backward(self, d_out):
        """Feed the gradient through the layers in reverse; return the gradient for the input."""
        # ✍️ Your code here (three lines)
        raise NotImplementedError

    def params(self):
        """Every layer's (parameter, gradient) pairs, in layer order, in one list."""
        # ✍️ Your code here (one to three lines)
        raise NotImplementedError

In [ ]:
def run_sequential(model_class, X, d_out):
    """Build Linear(2, 4), ReLU, Linear(4, 3) from a fixed seed, run forward and backward, return the results."""
    rng_check = np.random.default_rng(3)
    model = model_class(Linear(2, 4, rng_check), ReLU(), Linear(4, 3, rng_check))
    out = model.forward(X)
    d_in = model.backward(d_out)
    return out, d_in, model.params()


X_seq = np.random.default_rng(4).normal(size=(5, 2))
d_seq = np.random.default_rng(5).normal(size=(5, 3))
Sequential = practice_class("Sequential", Sequential, SequentialReference, run_sequential, cases=[
    (X_seq, d_seq),
    (X_seq[:1], d_seq[:1]),    # edge case: a batch of one
])

**Expected behaviour.** The demo applies two steps in order and prints 30. Your `Sequential` passes both cases: its output has shape `(5, 3)`, the gradient for the input has shape `(5, 2)`, and `params()` returns four pairs, `(W, dW)` and `(b, db)` for each `Linear` and none for the `ReLU`.

Two small pieces complete the framework, written for you. `SoftmaxCrossEntropy` combines the softmax and the loss in one object, as frameworks do, and its `backward` returns $(P - Y)/n$. `sgd_update` is the optimiser: it updates every parameter **in place**, which is what an optimiser's `step()` does.

In [ ]:
class SoftmaxCrossEntropy:
    """Softmax and mean cross-entropy in one object, the way frameworks combine them."""

    def forward(self, Z, y):
        self.probs, self.y = softmax(Z), y
        return cross_entropy(self.probs, y)

    def backward(self):
        return softmax_cross_entropy_grad(self.probs, self.y)


def sgd_update(params, lr):
    """Update every parameter in place: p -= lr * gradient, like an optimiser's step()."""
    for param, grad in params:
        param -= lr * grad

Now train with your classes. The training loop reads almost exactly like a PyTorch one: forward, loss, backward, step. All three `Linear` layers draw from one generator seeded with `SEED`, in the same order as `init_network`, so the model starts from the same weights as your functional network, and `minibatch_indices` gives it the same batches.

In [ ]:
layer_rng = np.random.default_rng(SEED)
model = Sequential(Linear(2, 64, layer_rng), ReLU(), Linear(64, 64, layer_rng), ReLU(), Linear(64, 3, layer_rng))
loss_fn = SoftmaxCrossEntropy()
started = time.perf_counter()
for epoch in range(EPOCHS):
    for batch in minibatch_indices(len(X_train), BATCH_SIZE, SEED + epoch):
        loss = loss_fn.forward(model.forward(X_train[batch]), y_train[batch])
        model.backward(loss_fn.backward())
        sgd_update(model.params(), LR)
class_seconds = time.perf_counter() - started
class_test_accuracy = float(np.mean(model.forward(X_test).argmax(axis=1) == y_test))
linear_layers = [layer for layer in model.layers if hasattr(layer, "W")]
largest_gap = max(float(np.abs(layer.W - trained["W"]).max()) for layer, trained in zip(linear_layers, numpy_network))
print(f"class-based model: test accuracy {class_test_accuracy:.3f} in {class_seconds:.2f} s")
print(f"functional model:  test accuracy {numpy_test_accuracy:.3f} in {numpy_seconds:.2f} s")
print(f"largest difference between the two models' weights: {largest_gap:.1e}")

**Expected behaviour.** The same test accuracy and a weight difference of 0, or at most around $10^{-15}$: your objects perform exactly the same arithmetic as your functions, only organised differently. The class-based version updates its arrays in place, as frameworks do, instead of building a new network at every step.

## 6 · Put it together

Three claims, checked: your layer classes reproduce your functional network, their `backward` methods agree with numerical gradients (notebook 06's `numerical_gradient`), and the benchmark learned the task.

In [ ]:
# 1. The object-style model is the functional model, organised differently.
assert largest_gap < 1e-9, "the class-based model should end with the functional model's weights"
assert abs(class_test_accuracy - numpy_test_accuracy) <= 1 / len(y_test), "both models should classify the test set alike"

# 2. Every parameter's gradient agrees with a central-difference estimate.
check_rng = np.random.default_rng(7)
tiny = Sequential(Linear(2, 5, check_rng), ReLU(), Linear(5, 3, check_rng))
tiny_loss = SoftmaxCrossEntropy()
X_tiny, y_tiny = X_train[:8], y_train[:8]
tiny_loss.forward(tiny.forward(X_tiny), y_tiny)
tiny.backward(tiny_loss.backward())
for param, grad in tiny.params():
    numeric = numerical_gradient(lambda _: SoftmaxCrossEntropy().forward(tiny.forward(X_tiny), y_tiny), param)
    assert relative_error(grad, numeric) < 1e-6, "a layer's backward disagrees with the numerical gradient"

# 3. The benchmark network learned the spirals (measured: about 0.98).
assert numpy_test_accuracy >= 0.90, f"expected at least 0.90 test accuracy, got {numpy_test_accuracy:.3f}"
print("✅ Your mini-framework matches your functions exactly, its gradients check out, and the model learned.")

## Exercises

### Exercise 1 · ✍️ A `Sigmoid` layer

Write a `Sigmoid` layer class in the same style. `forward(Z)` stores `self.out = sigmoid(Z)` and returns it; `backward(dA)` returns `dA * self.out * (1 - self.out)`, because $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$; `params()` returns `[]`. Then build `Sequential(Linear(2, 64, rng), Sigmoid(), Linear(64, 3, rng))` and train it: how does it compare with the ReLU version?

In [ ]:
class SigmoidReference:
    def forward(self, Z):
        self.out = sigmoid(Z)
        return self.out

    def backward(self, dA):
        return dA * self.out * (1 - self.out)

    def params(self):
        return []

In [ ]:
class Sigmoid:
    """The logistic function as a layer."""

    def forward(self, Z):
        # ✍️ Your code here
        raise NotImplementedError

    def backward(self, dA):
        # ✍️ Your code here
        raise NotImplementedError

    def params(self):
        return []

In [ ]:
Sigmoid = practice_class("Sigmoid", Sigmoid, SigmoidReference, run_activation, cases=[
    (Z_check, np.ones_like(Z_check)),
    (np.array([[-800.0, 0.0, 800.0]]), np.ones((1, 3))),    # extreme inputs must not overflow
])

### Exercise 2 · Open question: an optimiser with memory

Write a `MomentumSGD` class whose `step(params)` keeps one velocity array per parameter, $v \leftarrow \beta v - \eta g$, and updates $p \leftarrow p + v$. Where should the velocities live, given that `params()` returns the same arrays at every step? Compare its learning curve with `sgd_update` for $\beta = 0.9$.

### Exercise 3 · Open question: float32 (in Colab)

In the PyTorch cell, drop `dtype=torch.float64` and convert the data with `.float()` and the labels with `.long()`. How large is the weight difference from your NumPy network after 100 epochs now? Does the test accuracy change? Why do frameworks default to 32-bit, or even 16-bit, numbers?

### Exercise 4 · Open question: batch size and speed

Run the NumPy and scikit-learn comparisons again with `BATCH_SIZE = 8` and then 256 (adjust the learning rate if training fails). How do the training times change, and what does that say about where the time goes in section 4?

## Common errors and debugging

- **`RuntimeError: mat1 and mat2 must have the same dtype`.** In PyTorch the data and the model must have the same precision; NumPy arrays become `float64` tensors, while models default to `float32`.
- **Weights that look transposed.** `torch.nn.Linear` stores `weight` as `(n_out, n_in)`; your `W` is `(n_in, n_out)`.
- **Gradients that keep growing in PyTorch.** You forgot `optimizer.zero_grad()`, so each `backward()` added to the old gradients.
- **Softmax applied twice.** `torch.nn.functional.cross_entropy` expects logits; passing probabilities trains badly without any error.
- **`AttributeError: ... has no attribute 'X'`.** `backward` was called before `forward`, so the layer has nothing remembered.
- **Parameters that never change.** `param = param - lr * grad` only rebinds a local name; an in-place update needs `param -= lr * grad`.
- **Stale gradients from `params()`.** Ask for the pairs after `backward`: each call to `backward` stores new gradient arrays.

## Summary

scikit-learn and PyTorch run the same mathematics you wrote. scikit-learn wraps a standard network in four lines, with similar accuracy and speed on this problem and much less control. PyTorch's autograd computes exactly your hand-derived gradients, to within float64 rounding, for any model you can write; its real advantages, optimised kernels and accelerators, appear when models are large enough for arithmetic rather than overhead to dominate. Your own `Linear`, `ReLU` and `Sequential` classes are the skeleton of every such framework, and they reproduce your functional network to the last digit.

## Next

**12 · Final project**: package your functions into your own `NeuralNetwork` class, train it with an honest train/validation/test protocol, save and reload it, and explain what it can and cannot do.